<a href="https://colab.research.google.com/github/albormeha133/sustainspringhackathon/blob/main/Hackathon%20-%20Question%203%20script.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import xgboost as xgb

# Load data
Pesticides = pd.read_csv("/content/pesticides.csv")
Yield = pd.read_csv("/content/yield.csv")
Rainfall = pd.read_csv("/content/rainfall.csv")
Temperature = pd.read_csv("/content/temp.csv")

# Debugging: Print original columns of Rainfall
print("Rainfall original columns:", Rainfall.columns.tolist())

# Rename columns for Temperature
if "country" in Temperature.columns:
    Temperature = Temperature.rename(columns={"country": "Area"})
if "year" in Temperature.columns:
    Temperature = Temperature.rename(columns={"year": "Year"})
if "Value" in Temperature.columns:
    Temperature = Temperature.rename(columns={"Value": "avg_temp"})

# Rename columns for Rainfall
# The output showed ' Area' has a leading space, so let's handle that first
if ' Area' in Rainfall.columns:
    Rainfall = Rainfall.rename(columns={' Area': 'Area'})
# Assuming 'country' and 'year' might be present based on the error and similarity with Temperature
if "country" in Rainfall.columns:
    Rainfall = Rainfall.rename(columns={"country": "Area"})
elif "Entity" in Rainfall.columns: # Added check for 'Entity' as alternative to 'country'
    Rainfall = Rainfall.rename(columns={"Entity": "Area"})
if "year" in Rainfall.columns:
    Rainfall = Rainfall.rename(columns={"year": "Year"})
if "average_rain_fall_mm_per_year" in Rainfall.columns:
    Rainfall = Rainfall.rename(columns={"average_rain_fall_mm_per_year": "Average_Rainfall"})
elif "Value" in Rainfall.columns:
    Rainfall = Rainfall.rename(columns={"Value": "Average_Rainfall"})

# Rename 'Value' from Yield to 'Value_yield' explicitly before merging
Yield = Yield.rename(columns={"Value": "Value_yield"})

# Merge data
data = Yield.merge(
    Pesticides[["Area", "Year", "Value"]].rename(columns={"Value": "Value_pest"}), # Rename Pesticides' Value here
    on=["Area", "Year"],
    how="left"
)

# 'Value' from Yield is now 'Value_yield', so no further rename needed for yield data

data = data.merge(
    Rainfall[["Area", "Year", "Average_Rainfall"]],
    on=["Area", "Year"],
    how="left"
)

data = data.merge(
    Temperature[["Area", "Year", "avg_temp"]],
    on=["Area", "Year"],
    how="left"
)

data = data.dropna(subset=["Value_yield", "Area", "Year"])


Rainfall original columns: [' Area', 'Year', 'average_rain_fall_mm_per_year']


In [3]:
results = []

for country_x in ["India", "Brazil", "Mexico", "Pakistan", "Canada"]:
  train_data = data[data["Year"] <= 2008].copy()
  test_data = data[data["Year"] > 2008].copy()

  global_train = train_data.copy()
  global_test = test_data[test_data["Area"] == country_x].copy()

  local_train = train_data[train_data["Area"] == country_x].copy()
  local_test = test_data[test_data["Area"] == country_x].copy()

  if len(global_test) == 0 or len(local_train) == 0 or len(local_test) == 0:
          results.append({
              "Country": country_x,
              "Global MAE": None,
              "Local MAE": None,
              "Winner": "Not enough data"
          })
          continue

  print("Global train:", len(global_train))
  print("Global test:", len(global_test))
  print("Local train:", len(local_train))
  print("Local test:", len(local_test))

Global train: 95451
Global test: 904
Local train: 8448
Local test: 904
Global train: 95451
Global test: 522
Local train: 4752
Local test: 522
Global train: 95451
Global test: 344
Local train: 3072
Local test: 344
Global train: 95451
Global test: 336
Local train: 3024
Local test: 336
Global train: 95451
Global test: 132
Local train: 1152
Local test: 132


In [4]:
feature_cols = ["Value_pest", "Year", "Item", "Area", "Average_Rainfall", "avg_temp"]

# Global model
X_global_train = pd.get_dummies(global_train[feature_cols])
X_global_test = pd.get_dummies(global_test[feature_cols])
X_global_test = X_global_test.reindex(columns=X_global_train.columns, fill_value=0)

y_global_train = global_train["Value_yield"]
y_global_test = global_test["Value_yield"]

global_model = xgb.XGBRegressor(
    n_estimators=100,
    max_depth=4,
    random_state=42
)
global_model.fit(X_global_train, y_global_train)
global_preds = global_model.predict(X_global_test)

# Local model
X_local_train = pd.get_dummies(local_train[feature_cols])
X_local_test = pd.get_dummies(local_test[feature_cols])
X_local_test = X_local_test.reindex(columns=X_local_train.columns, fill_value=0)

y_local_train = local_train["Value_yield"]
y_local_test = local_test["Value_yield"]

local_model = xgb.XGBRegressor(
    n_estimators=100,
    max_depth=4,
    random_state=42
)
local_model.fit(X_local_train, y_local_train)
local_preds = local_model.predict(X_local_test)

In [5]:
global_mae = float((abs(y_global_test.to_numpy() - global_preds)).mean())
local_mae = float((abs(y_local_test.to_numpy() - local_preds)).mean())

results.append({
    "Country": country_x,
    "Global MAE": round(global_mae, 2),
    "Local MAE": round(local_mae, 2),
    "Winner": "Local model" if local_mae < global_mae else "Global model"
})

print("Country:", country_x)
print("Global MAE:", round(global_mae, 2))
print("Local MAE:", round(local_mae, 2))

if local_mae < global_mae:
    print("Winner: Local model")
else:
    print("Winner: Global model")

Country: Canada
Global MAE: 27152.66
Local MAE: 2862.26
Winner: Local model
